# Session 2 lab — Data storage systems

One month of New York City yellow taxi trips, written down four different ways.
Same rows, same columns, same numbers — four files.

You will ask each file the same three questions and record two things: **how many
bytes the file forces you to read**, and how long it took. Then you will do the
same in PostgreSQL, with and without an index.

**You do not need to write any SQL.** Every query is already here. Run the cells
in order, top to bottom.

The claim the whole lab tests: **the cost of a question is decided by how the
data is stored, not by how you phrase it.** Part A varies the layout of the file,
Part B varies whether there is an index. Same data, same questions, throughout.

## Your results

### Part A — four representations

Times differ between laptops; the comparison between rows is the point.

| Format  | Size on disk | Q1 whole table | Q2 avg fare by hour | Q3 one trip |
|---------|--------------|----------------|---------------------|-------------|
| CSV     |              |                |                     |             |
| JSON    |              |                |                     |             |
| Parquet |              |                |                     |             |
| SQLite  |              |                |                     |             |

### Part B — PostgreSQL

|                       | Without index | With index |
|-----------------------|---------------|------------|
| One trip lookup       |               |            |
| Regulator's aggregate |               |            |

### Your conclusion

Three sentences: for the workload described in class, which system would you
choose, and what does it cost you?

## Part A — Four representations (12 min)

Every question below goes through the same engine, DuckDB, so what changes
between the rows of each table is the file, not the tool.

Alongside each time you get **must read** — the bytes that format gives the
engine no way to avoid touching. That number is a property of the file, not of
your laptop, and it is the one to pay attention to.

**Why DuckDB, and not pandas.** It is the one engine that can read all four of
these through a single interface, which is what makes the file the only thing
changing between the rows of each table. It also queries files where they lie,
with no import step — in Part A the file *is* the database. And it needs no
server, so there is nothing to set up. Spark, BigQuery and Snowflake treat
Parquet the same way; DuckDB is the one that fits on a laptop.

**What you should see.** Three of the four files answer every question by reading
all of themselves. Only Parquet's numbers move: several times smaller on disk,
then smaller again once you only want two columns — and then, for a single row,
about the same as the two-column question. That last one is its own lesson:
without an index, fetching one trip costs roughly what summarising a million of
them costs. Your bytes should match your neighbour's exactly, on any laptop; your
seconds will not match anyone's.

In [1]:
import time
from pathlib import Path

import duckdb
import pyarrow.parquet as pq


def find_data() -> Path:
    candidates = (
        Path("../../data/nyc-taxi"),  # from this notebook's own folder
        Path("data/nyc-taxi"),  # from the repository root
        Path("/app/data/nyc-taxi"),  # from anywhere inside the container
    )
    for candidate in candidates:
        if (candidate / "trips.parquet").exists():
            return candidate.resolve()
    raise SystemExit("No data. Run scripts/download_nyc_taxi.py first — see README.md")


DATA = find_data()
duckdb.sql("LOAD sqlite")  # lets DuckDB read a SQLite file too

FILES = {
    "CSV": DATA / "trips.csv",
    "JSON": DATA / "trips.jsonl",
    "Parquet": DATA / "trips.parquet",
    "SQLite": DATA / "trips.sqlite",
}

# The same table, named four ways, for the queries below.
SOURCES = {
    "CSV": f"read_csv('{FILES['CSV']}')",
    "JSON": f"read_json_auto('{FILES['JSON']}')",
    "Parquet": f"'{FILES['Parquet']}'",
    "SQLite": f"sqlite_scan('{FILES['SQLite']}', 'trips')",
}

for name, source in SOURCES.items():
    rows = duckdb.sql(f"SELECT count(*) FROM {source}").fetchone()[0]
    size = FILES[name].stat().st_size / 1_000_000
    print(f"{name:<8} {size:7,.0f} MB on disk   {rows:,} rows")

CSV          112 MB on disk   1,000,000 rows
JSON         445 MB on disk   1,000,000 rows
Parquet       35 MB on disk   1,000,000 rows
SQLite       122 MB on disk   1,000,000 rows


Same rows in every one of them, and nowhere near the same size. CSV and JSON
spell every number out as text, and JSON repeats all twenty column names on
every single row. Parquet stores each column separately, typed
and compressed.

Next, the machinery for the three questions — how much of a file must be read,
and how long the query takes.

In [2]:
# A Parquet file carries an index of itself: where each column sits, how big it
# is, and the smallest and largest value in it. We read that index — not the data
# — to work out how little of the file a query can get away with touching.
PARQUET = pq.ParquetFile(FILES["Parquet"])
META = PARQUET.metadata
CHUNKS = [META.row_group(i) for i in range(META.num_row_groups)]
COLUMN_AT = {CHUNKS[0].column(i).path_in_schema: i for i in range(META.num_columns)}

ALL_COLUMNS = list(COLUMN_AT)
FARE_COLUMNS = ["tpep_pickup_datetime", "fare_amount"]
OTHER_COLUMNS = [column for column in ALL_COLUMNS if column != "trip_id"]

TRIP_ID = META.num_rows // 2  # any trip; there is nothing special about this one


def parquet_bytes(chunks: list[pq.RowGroupMetaData], columns: list[str]) -> int:
    """Compressed size of those columns, inside those chunks."""
    return sum(
        chunk.column(COLUMN_AT[column]).total_compressed_size
        for chunk in chunks
        for column in columns
    )


def chunk_holding(trip_id: int) -> pq.RowGroupMetaData:
    """Which chunk holds this trip — found by looking, because nothing says.

    The rows were shuffled before the files were written, so an id tells you
    nothing about where its row ended up. This is the honest situation: the file
    has no idea where anything is.
    """
    for index, chunk in enumerate(CHUNKS):
        ids = PARQUET.read_row_group(index, columns=["trip_id"])["trip_id"]
        if trip_id in ids.to_pylist():
            return chunk
    raise LookupError(f"no chunk holds trip {trip_id}")


# What each file cannot avoid reading, per question. CSV, JSON and SQLite give the
# same answer to all three — all of it — because a row-oriented file offers the
# reader nothing to skip.
WHOLE_FILE = {name: path.stat().st_size for name, path in FILES.items()}
MUST_READ = {
    "Q1": WHOLE_FILE | {"Parquet": parquet_bytes(CHUNKS, ALL_COLUMNS)},
    "Q2": WHOLE_FILE | {"Parquet": parquet_bytes(CHUNKS, FARE_COLUMNS)},
    # Finding one trip costs the id column of every chunk, and then the rest of
    # the single chunk it turned out to be in.
    "Q3": WHOLE_FILE
    | {
        "Parquet": parquet_bytes(CHUNKS, ["trip_id"])
        + parquet_bytes([chunk_holding(TRIP_ID)], OTHER_COLUMNS)
    },
}

print(f"we will look for trip_id {TRIP_ID:,}, one of {META.num_rows:,}")
print(f"Parquet holds {META.num_columns} columns in {META.num_row_groups} chunks")

we will look for trip_id 500,000, one of 1,000,000
Parquet holds 20 columns in 10 chunks


In [3]:
def run(sql: str) -> None:
    duckdb.execute(sql).fetchall()  # fetch, so we wait for the whole answer


def seconds(sql: str) -> float:
    started = time.perf_counter()
    run(sql)
    return time.perf_counter() - started


def ask(question: str, query: str, must_read: dict[str, int]) -> None:
    """Put one question to all four files and report bytes, then time.

    `query` is written once with {source} where the file goes. Each file gets one
    warm-up run, then three timed runs, and we keep the fastest of the three.
    """
    print(f"{question}\n")
    for name, source in SOURCES.items():
        sql = query.format(source=source, trip=TRIP_ID)
        run(sql)
        best = min(seconds(sql) for _ in range(3))
        megabytes = must_read[name] / 1_000_000
        print(f"  {name:<8} must read {megabytes:7,.1f} MB   {best:7.3f} s")
    print()

### Q1 — Give me the whole table

The question a script asks when it starts by loading "the data" without thinking
about it. Every format has to hand over all twenty columns and every row, so
nobody can read less than everything.

Do not be surprised if JSON finishes this one faster than CSV despite reading
four times as many bytes — DuckDB simply throws more threads at it. **The bytes
are a property of the file; the seconds are a property of the reader.** That is
why the bytes are the column to trust.

In [4]:
ask(
    "Q1  the whole table",
    "CREATE OR REPLACE TABLE everything AS SELECT * FROM {source}",
    MUST_READ["Q1"],
)

Q1  the whole table

  CSV      must read   112.3 MB     0.348 s
  JSON     must read   445.3 MB     0.305 s
  Parquet  must read    34.5 MB     0.203 s
  SQLite   must read   122.4 MB     0.724 s



### Q2 — The regulator's question

> *What was the average fare, by hour of the day, across the whole month?*

Still every row — but only two of the twenty columns. Watch what happens to
**must read**.

In [5]:
REGULATOR = """
SELECT date_part('hour', tpep_pickup_datetime::TIMESTAMP) AS hour,
       round(avg(fare_amount), 2) AS avg_fare,
       count(*) AS trips
FROM {source}
GROUP BY hour
ORDER BY hour
"""

ask("Q2  average fare by hour of day", REGULATOR, MUST_READ["Q2"])

duckdb.sql(REGULATOR.format(source=SOURCES["Parquet"])).df().head(24)

Q2  average fare by hour of day

  CSV      must read   112.3 MB     0.134 s
  JSON     must read   445.3 MB     0.123 s
  Parquet  must read     9.8 MB     0.004 s
  SQLite   must read   122.4 MB     0.037 s



,hour,avg_fare,trips
0,0,20.21,23952
1,1,18.21,16320
2,2,17.84,12120
3,3,19.57,8511
4,4,23.51,5466
5,5,26.99,6954
6,6,23.01,14956
7,7,19.07,29376
8,8,17.66,40834
9,9,18.02,46011


Three of the four read exactly what they read before: they store rows, so the two
columns you want are interleaved with the eighteen you do not, and skipping them
is not possible. Parquet reads a fraction of itself, because each column sits in
its own chunk.

SQLite gets noticeably quicker here all the same — it still walks every page, but
it can skip *decoding* the columns nobody asked for. Fewer bytes read, no; less
work done, yes.

**Careful about what DuckDB is contributing.** Skipping eighteen columns is
something the *file* makes possible; DuckDB is just an engine that knows how to
take the offer. Spark, BigQuery and Snowflake all work this way — DuckDB is the
one that fits on a laptop. Two things the seconds do not tell you: DuckDB has an
unusually good CSV reader, so CSV looks better here than it will in most tools,
and `pd.read_parquet(path)` without `columns=` throws Parquet's advantage away
entirely. Part of the gap is also encoding rather than layout — Parquet stores a
real timestamp, while CSV has to parse `"2024-01-15 08:23:41"` out of text.

The bytes belong to the file. The seconds belong to whoever is reading it.

This is the whole argument for columnar storage, and it is why analytical
questions on Parquet are cheap.

### Q3 — One trip

A single row, by `trip_id`. This is the question none of these files is built for.

The rows were **shuffled before the files were written**, so an id says nothing
about where its row sits — which is the normal situation for whatever you happen
to search by. CSV, JSON and SQLite read everything, checking as they go. Parquet
reads the id column out of all ten chunks to work out where the trip is, then
reads the rest of that one chunk: far fewer bytes, but still a scan. It scans one
narrow column instead of every byte.

Had we written the file *sorted* by `trip_id`, each chunk's recorded smallest and
largest id would have ruled out nine chunks without reading them at all. That is
what "clustering" means in BigQuery or Delta Lake — and it is a decision made when
the file is written, for one chosen column, not for every column at once.

In [6]:
ask(
    f"Q3  the one trip with id {TRIP_ID:,}",
    "SELECT * FROM {source} WHERE trip_id = {trip}",
    MUST_READ["Q3"],
)

Q3  the one trip with id 500,000

  CSV      must read   112.3 MB     0.145 s
  JSON     must read   445.3 MB     0.131 s
  Parquet  must read     9.7 MB     0.005 s
  SQLite   must read   122.4 MB     0.523 s



**Fill in the Part A table at the top now.**

Then answer this: which file would you hand to a colleague who has to answer the
regulator's question every morning, and which to one who has to open it in Excel?

Notice what is still missing. Every one of these files answers "where is trip
550,000?" by reading its way through the data until it turns up. Parquet reads
much less of itself than the others, but none of them can *find* anything. That is
what Part B is about.

## Part B — Reading every page, or the index (10 min)

Same data, now inside a database server: PostgreSQL, running in the same
`docker compose` that serves this notebook.

An index is a second structure beside the table that lets the server find rows
without looking at all of them. The interesting part is not that indexes make
things faster — it is *which* things.

**Why PostgreSQL here.** Part A left us stuck: none of those files can *find* a
row, only read all of themselves looking for it. Fixing that needs a system that
maintains an index and will tell you whether it used one — that is
`EXPLAIN ANALYZE`. PostgreSQL is also the opposite design to Parquet: rows stored
together, built for many small reads and writes rather than for scanning a
column.

**What you should see.** The single-row lookup gets faster by something like a
thousandfold, and the regulator's aggregate does not budge.

In [7]:
import io

import pandas as pd
import psycopg

POSTGRES = "host=postgres port=5432 user=labs password=labs dbname=labs"

COLUMNS = [
    "trip_id",
    "tpep_pickup_datetime",
    "trip_distance",
    "fare_amount",
    "tip_amount",
    "total_amount",
]

# Row-by-row INSERT of a million rows would take minutes. COPY is the bulk path.
trips = pd.read_parquet(FILES["Parquet"], columns=COLUMNS)
buffer = io.StringIO()
trips.to_csv(buffer, index=False, header=False)
buffer.seek(0)

connection = psycopg.connect(POSTGRES, autocommit=True)
cursor = connection.cursor()

started = time.perf_counter()
cursor.execute("DROP TABLE IF EXISTS trips")
cursor.execute("""
    CREATE TABLE trips (
        trip_id              integer,
        tpep_pickup_datetime timestamp,
        trip_distance        double precision,
        fare_amount          numeric,
        tip_amount           numeric,
        total_amount         numeric
    )
""")
with cursor.copy("COPY trips FROM STDIN WITH (FORMAT csv)") as copy:
    copy.write(buffer.read())
cursor.execute("ANALYZE trips")
print(f"{len(trips):,} rows copied in {time.perf_counter() - started:.1f} s")

1,000,000 rows copied in 0.9 s


### How to read a query plan

`EXPLAIN ANALYZE` shows how PostgreSQL decided to run a query. Two phrases are
worth recognising:

- **Seq Scan** — read every row in the table and check each one. (Sometimes
  *Parallel Seq Scan*: the same thing, split across CPU cores.)
- **Index Scan** — go to the index, jump straight to the rows that match.

In [8]:
LOOKUP = f"SELECT * FROM trips WHERE trip_id = {TRIP_ID}"

REGULATOR_SQL = """
SELECT date_part('hour', tpep_pickup_datetime) AS hour,
       round(avg(fare_amount), 2) AS avg_fare,
       count(*) AS trips
FROM trips
GROUP BY hour
ORDER BY hour
"""


def explain(label: str, sql: str) -> float:
    """Time one query and report the scan PostgreSQL chose.

    Four runs: one to warm the cache, then three measured, keeping the fastest.
    A single measurement of something this quick is mostly noise.
    """
    cursor.execute(sql)
    cursor.fetchall()

    times, scan = [], "?"
    for _ in range(3):
        cursor.execute("EXPLAIN (ANALYZE, COSTS OFF) " + sql)
        plan = [row[0] for row in cursor.fetchall()]
        scan = next((line.strip() for line in plan if "Scan" in line), "?")
        times.append(
            next(
                float(line.split()[2])
                for line in plan
                if line.startswith("Execution Time:")
            )
        )

    best = min(times)
    print(f"{label:<26} {best:9.2f} ms   {scan}")
    return best

### B1 — Without an index

There is no index on `trip_id` yet. Watch what PostgreSQL has to do to find one
row — and note that it is the same thing Part A's files had to do.

In [9]:
lookup_before = explain("lookup, no index", LOOKUP)
aggregate_before = explain("aggregate, no index", REGULATOR_SQL)

lookup, no index               13.83 ms   ->  Parallel Seq Scan on trips (actual time=7.977..10.438 rows=0 loops=3)
aggregate, no index           251.76 ms   ->  Seq Scan on trips (actual time=1.831..69.187 rows=1000000 loops=1)


### B2 — Add the index, ask again

One line. It costs disk space, and time on every write, and in exchange the
server gets a way to find rows by `trip_id` without reading the table.

In [10]:
cursor.execute("CREATE INDEX ON trips (trip_id)")

lookup_after = explain("lookup, with index", LOOKUP)
aggregate_after = explain("aggregate, with index", REGULATOR_SQL)


def verdict(before: float, after: float) -> str:
    """Anything under 2x here is noise, not a result."""
    ratio = before / after
    return f"{ratio:>6.0f}x faster" if ratio >= 2 else "     unchanged"


print()
print("=" * 62)
print(
    f"  one trip lookup       {lookup_before:8.2f} ->{lookup_after:8.2f} ms"
    f"   {verdict(lookup_before, lookup_after)}"
)
print(
    f"  regulator's aggregate {aggregate_before:8.2f} ->{aggregate_after:8.2f} ms"
    f"   {verdict(aggregate_before, aggregate_after)}"
)
print("=" * 62)

lookup, with index              0.00 ms   Index Scan using trips_trip_id_idx on trips (actual time=0.003..0.003 rows=1 loops=1)
aggregate, with index         258.97 ms   ->  Seq Scan on trips (actual time=2.038..75.934 rows=1000000 loops=1)

  one trip lookup          13.83 ->    0.00 ms     3458x faster
  regulator's aggregate   251.76 ->  258.97 ms        unchanged


The index answers *where is one particular trip* without touching the table. It
does nothing at all for the average fare per hour, because that question needs
every row anyway — so PostgreSQL ignores the index and scans the table, exactly
as it did before.

An index is not a speed setting. It is an answer to one specific question.

And that non-result is the bridge back to Part A. A row store with the right
index is unbeatable at *fetch me this one trip*, and no index will save it from
*summarise all million rows* — it still reads every page. That question belongs to
the columnar side of the session, which is why both kinds of system exist and why
firms run both.

**Fill in the Part B table.** Then: if someone told you to "add an index to make
the dashboard faster", what would you need to know first?

## Part C — The lost payout (optional)

If you have finished both parts and want to see the other half of the story —
what happens when two people write to the same row at the same moment — open
**`PART_C.md`**. It needs two terminals side by side, which is why it is not in
this notebook.

Two payouts arrive for one driver. Run them the ordinary way and the balance ends
at **145.00** where the driver is owed **185.00** — €40 gone, with no error
anywhere. Wrap them in transactions and you get 185.00, at the price of one
terminal visibly sitting there waiting.

Nothing later in the course depends on it.